# bob, explained - Episode 13: Every error we hit, and the rule it became

Every error we hit, and the rule it became

Run `!pip install manim` and `from manim import *` once first, then this cell. Start at `-ql`; the class names listed at the top of the cell render a single section.


In [ ]:
%%manim -qm Ep13Errors
# =============================================================================
#  bob, explained - EPISODE 13: Every error we hit, and the rule it became
#  Every error we hit, and the rule it became
#
#  GENERATED by docs/manim/build.py from docs/manim/parts/. Do not edit here.
#
#  Prerequisite (once per notebook, in a cell of its own):
#      !pip install manim
#      from manim import *
#
#  Quality on the magic line above:  -ql draft   -qm medium   -qh 1080p60
#
#  Render one section instead of the whole episode by putting any of these
#  class names on the magic line:
#      E13S1Intro
#      E13S2Tcl
#      E13S3Guard
#      E13S4Vivado
#      E13S5Board
#      E13S6Subtle
#      E13S7Rules
#      E13S8Files
# =============================================================================

# =============================================================================
#  shared prelude - palette, helpers and the BobScene base class.
#  docs/manim/build.py pastes this into the top of every episode cell.
# =============================================================================

from manim import *
import numpy as np

# ---------------------------------------------------------------- palette ----
BG    = "#11121a"
INK   = "#e8e8ea"
DIM   = "#8b93a7"
C_PY  = "#7aa2f7"   # blue    - Python / tools / the device description
C_VPR = "#f7768e"   # red     - VPR / external tools
C_RTL = "#9ece6a"   # green   - hardware, Verilog, things on the die
C_BIT = "#e0af68"   # amber   - configuration bits, FASM, the bitstream
C_GRF = "#bb9af7"   # purple  - graphs, JTAG, protocol
C_ERR = "#ff7a93"   # pink    - bugs, refusals, errors
MONO  = "monospace"


# ---------------------------------------------------------------- helpers ----
def mono(s, size=22, color=INK):
    """One line of monospace text (Pango crashes on '', so blanks become ' ')."""
    return Text(s if s else " ", font=MONO, font_size=size, color=color)


def code_block(lines, size=20, color=INK):
    g = VGroup(*[mono(l, size, color) for l in lines])
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.14)
    return g


def panel(mob, color=DIM, pad=0.32, fill=0.06):
    r = SurroundingRectangle(mob, color=color, buff=pad)
    r.set_fill(color, opacity=fill)
    return VGroup(r, mob)


def chip(label, color, w=2.6, h=0.95, size=22, weight="NORMAL"):
    box = RoundedRectangle(width=w, height=h, corner_radius=0.14,
                           color=color, stroke_width=3)
    box.set_fill(color, opacity=0.12)
    txt = Text(label, font_size=size, color=INK, weight=weight, line_spacing=0.75)
    if txt.width > w - 0.3:
        txt.scale_to_fit_width(w - 0.3)
    if txt.height > h - 0.2:
        txt.scale_to_fit_height(h - 0.2)
    return VGroup(box, txt.move_to(box.get_center()))


def arrow(a, b, color=DIM, buff=0.15, sw=3):
    return Arrow(a, b, buff=buff, color=color, stroke_width=sw,
                 max_tip_length_to_length_ratio=0.18)


def mux_symbol(color=C_RTL, h=1.9, w=0.8):
    """Classic trapezoid multiplexer symbol."""
    p = Polygon([-w / 2,  h / 2, 0], [w / 2,  h / 2 - 0.3, 0],
                [ w / 2, -h / 2 + 0.3, 0], [-w / 2, -h / 2, 0],
                color=color, stroke_width=3)
    p.set_fill(color, opacity=0.14)
    return p


def bitcells(n, size=0.3, on=(), color=C_BIT, off_color=DIM):
    """A strip of n little squares; indices in `on` are filled."""
    g = VGroup()
    for i in range(n):
        s = Square(size, color=off_color, stroke_width=1.6)
        if i in on:
            s.set_stroke(color).set_fill(color, opacity=0.85)
        g.add(s)
    g.arrange(RIGHT, buff=0.035)
    return g


def fieldbar(fields, total_w=11.0, h=0.62, size=15):
    """
    fields: [(label, nbits, color), ...] -> one horizontal bar split to scale,
    each slice labelled above and its bit range below. Returns VGroup(bar, labels, ranges).
    """
    nbits = sum(f[1] for f in fields)
    bar, labs, rngs = VGroup(), VGroup(), VGroup()
    x, lo = -total_w / 2, 0
    for label, n, col in fields:
        w = max(total_w * n / nbits, 0.34)
        r = Rectangle(width=w, height=h, color=col, stroke_width=2)
        r.set_fill(col, opacity=0.28).move_to(np.array([x + w / 2, 0, 0]))
        bar.add(r)
        t = Text(label, font_size=size, color=col)
        if t.width > w * 1.9:
            t.scale_to_fit_width(max(w * 1.9, 0.5))
        t.next_to(r, UP, buff=0.14)
        labs.add(t)
        rt = mono(f"{lo}" if n == 1 else f"{lo}..{lo + n - 1}", size - 2, DIM)
        rt.next_to(r, DOWN, buff=0.12)
        if rt.width > w * 1.9:
            rt.scale_to_fit_width(max(w * 1.9, 0.5))
        rngs.add(rt)
        x += w
        lo += n
    return VGroup(bar, labs, rngs)


def filecard(path, role, color):
    """A small card naming a repo file and what it is."""
    t = mono(path, 17, color)
    r = Text(role, font_size=14, color=DIM)
    g = VGroup(t, r).arrange(DOWN, aligned_edge=LEFT, buff=0.08)
    box = SurroundingRectangle(g, color=color, buff=0.16)
    box.set_fill(color, opacity=0.07)
    return VGroup(box, g)


def mid(a, b):
    """midpoint, defined here so nothing depends on manim exporting space_ops."""
    return (a + b) / 2


def clear_all(sc, run_time=0.6):
    if sc.mobjects:
        sc.play(*[FadeOut(m) for m in sc.mobjects], run_time=run_time)


class BobScene(Scene):
    def setup(self):
        self.camera.background_color = BG

    def heading(self, text, kicker=None):
        t = Text(text, font_size=32, color=INK, weight="BOLD")
        t.to_corner(UL).shift(DOWN * 0.1)
        rule = Line(LEFT * 6.6, RIGHT * 6.6, color=DIM, stroke_width=1.5)
        rule.next_to(t, DOWN, buff=0.2).align_to(t, LEFT)
        g = VGroup(t, rule)
        self.play(FadeIn(t, shift=RIGHT * 0.3), Create(rule), run_time=0.7)
        if kicker:
            k = Text(kicker, font_size=19, color=DIM)
            if k.width > 13.0:
                k.scale_to_fit_width(13.0)
            k.next_to(rule, DOWN, buff=0.16).align_to(t, LEFT)
            g.add(k)
            self.play(FadeIn(k), run_time=0.4)
        return g

    def titlecard(self, number, title, subtitle):
        n = Text(number, font_size=26, color=C_BIT, weight="BOLD")
        t = Text(title, font_size=60, color=INK, weight="BOLD")
        s = Text(subtitle, font_size=26, color=DIM)
        if t.width > 12.5:
            t.scale_to_fit_width(12.5)
        if s.width > 12.5:
            s.scale_to_fit_width(12.5)
        g = VGroup(n, t, s).arrange(DOWN, buff=0.4)
        self.play(FadeIn(n), run_time=0.4)
        self.play(Write(t), run_time=1.1)
        self.play(FadeIn(s, shift=UP * 0.2), run_time=0.7)
        self.wait(1.6)
        self.play(FadeOut(g), run_time=0.6)

    def files_used(self, inputs, generated, verified):
        """Closing card: what this episode's topic is built from and checked by."""
        self.heading("Files", "what this part is written in, what is generated, and what proves it")
        cols = []
        for title, items, col in (("written by hand", inputs, C_RTL),
                                  ("generated", generated, C_PY),
                                  ("verified by", verified, C_BIT)):
            head = Text(title, font_size=21, color=col, weight="BOLD")
            cards = VGroup(*[filecard(p, r, col) for p, r in items])
            cards.arrange(DOWN, aligned_edge=LEFT, buff=0.18)
            g = VGroup(head, cards).arrange(DOWN, aligned_edge=LEFT, buff=0.28)
            cols.append(g)
        row = VGroup(*cols).arrange(RIGHT, buff=0.7, aligned_edge=UP)
        if row.width > 13.2:
            row.scale_to_fit_width(13.2)
        row.next_to(self.mobjects[1], DOWN, buff=0.55).set_x(0)
        for c in cols:
            self.play(FadeIn(c, shift=UP * 0.2), run_time=0.7)
        self.wait(2.4)

# =============================================================================
#  EPISODE 13 - Every error we hit, and the rule it became
# =============================================================================

def s1_intro(sc):
    n = Text("42", font_size=110, color=C_ERR, weight="BOLD").shift(UP * 0.8)
    t = Text("entries in the problems table", font_size=28, color=INK)
    t.next_to(n, DOWN, buff=0.4)
    sc.play(Write(n), run_time=0.9)
    sc.play(FadeIn(t), run_time=0.6)
    sc.wait(1.0)

    k = Text("Almost every one of them became a rule, a test, or both.",
             font_size=26, color=C_BIT)
    k.next_to(t, DOWN, buff=0.7)
    sc.play(FadeIn(k), run_time=0.8)
    sc.wait(1.6)
    sc.play(FadeOut(VGroup(n, t, k)), run_time=0.6)

    sc.heading("The worst kind of bug", "not the one that breaks - the one that passes")
    kinds = code_block([
        "a constraint file that was silently skipped, so nothing was constrained",
        "a guard that could be deleted with every test still green",
        "a test whose stimulus never exercised the thing it claimed to check",
        "a mutation that stopped matching when the grid grew",
        "",
        "All four of these happened here. All four now have a test that catches them.",
    ], 21, INK)
    kinds[5].set_color(C_BIT)
    kinds.next_to(sc.mobjects[1], DOWN, buff=0.9).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l, shift=RIGHT * 0.15) for l in kinds], lag_ratio=0.22),
            run_time=2.4)
    sc.wait(2.0)


def s2_tcl(sc):
    sc.heading("1. Tcl in the XDC", "M0 - and it reported success the whole time")

    bad = code_block([
        "the old constraint file did this:",
        "",
        "    if {$top eq \"fpga4x4_top\"} {",
        "        create_clock -period 1000.0 -name tck [get_ports tck]",
        "    }",
    ], 21, C_ERR)
    bad[0].set_color(DIM)
    bad.shift(UP * 1.7)
    sc.play(FadeIn(bad), run_time=0.9)

    what = code_block([
        "Vivado's XDC parser rejects Tcl control flow. It does not stop the build -",
        "it prints a critical warning and SKIPS the line.",
        "",
        "So create_clock never ran. TCK had no clock. Nothing was analysed.",
        "And the report said: all constraints met.",
    ], 20, INK)
    what[3].set_color(C_ERR); what[4].set_color(C_ERR)
    what.next_to(bad, DOWN, buff=0.7).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in what], lag_ratio=0.2), run_time=2.0)

    rule = code_block([
        "RULE   the XDC is plain XDC.  tests/test_layout.py rejects Tcl commands in it.",
        "       Anything top-dependent lives in drc_waiver.tcl, which is real Tcl.",
        "       tests/test_reports.py checks the report says no_clock (0).",
    ], 19, C_RTL)
    rule.to_edge(DOWN, buff=0.4).set_x(0)
    sc.play(FadeIn(rule), run_time=0.9)
    sc.wait(2.2)


def s3_guard(sc):
    sc.heading("2. A guard that nobody tested", "M2 - which is why this project has mutation tests")

    story = code_block([
        "The configuration plane refuses a load whose bit count is wrong.",
        "There was a testbench for it. It passed.",
        "",
        "Then someone deleted the length check and ran the testbench again.",
        "It still passed.",
    ], 22, INK)
    story[4].set_color(C_ERR)
    story.shift(UP * 1.6)
    sc.play(FadeIn(story[0]), FadeIn(story[1]), run_time=0.9)
    sc.wait(0.8)
    sc.play(FadeIn(story[3]), run_time=0.6)
    sc.play(FadeIn(story[4]), run_time=0.6)
    sc.wait(1.0)

    fix = code_block([
        "make mutate now breaks each guard ON PURPOSE and requires a testbench to fail:",
        "",
        "   mutate_cfg.sh       5 mutants   CRC polynomial, CRC guard, length guard,",
        "                                   write key, start without commit",
        "   mutate_fabric.sh   25 mutants   GSR, GWE freeze, gce, routed CE, dividers,",
        "                                   BRAM modes, DSP cascade, mux encoding, carry",
        "   mutate_frames.sh   29 mutants   every frame-path refusal",
        "",
        "59 mutants. All killed. Two needed NEW scenarios before they could be.",
    ], 19, INK)
    fix[0].set_color(DIM)
    fix[8].set_color(C_BIT)
    fix.next_to(story, DOWN, buff=0.7).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in fix], lag_ratio=0.15), run_time=2.4)
    sc.wait(2.2)


def s4_vivado(sc):
    sc.heading("3. Vivado crashed four times", "M13 - and the fourth one restarted Windows")

    rows = [
        ("1", "closed during synthesis",
         "cfg[idx*128 +: 128] <= data over 4992 bits is a barrel shifter:\n"
         "~12k LUTs, 1.7 GB in yosys alone", C_ERR),
        ("2", "crashed again, same point",
         "keep_hierarchy on u_fabric made Vivado use set_disable_timing on\n"
         "the kept boundary while breaking timing loops", C_ERR),
        ("3", "same crash",
         "keep_hierarchy on the SMALL modules was enough to do it too", C_ERR),
        ("4", "same step, and Windows restarted",
         "a program crash does not restart Windows - the machine was under load,\n"
         "and the constant was XDC-driven loop breaking during synthesis", C_ERR),
        ("5", "built in 3.5 min at 2.0 GB",
         "per-frame write decode, fabric flattened, multicycles by clock,\n"
         "and USED_IN_SYNTHESIS false on the XDC", C_RTL),
    ]
    g = VGroup()
    for n, a, b, col in rows:
        num = mono(n, 20, col)
        t = Text(a, font_size=18, color=col)
        w = Text(b, font_size=15, color=DIM, line_spacing=0.8)
        g.add(VGroup(num, VGroup(t, w).arrange(DOWN, aligned_edge=LEFT, buff=0.08))
              .arrange(RIGHT, buff=0.35, aligned_edge=UP))
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.24)
    if g.height > 5.0:
        g.scale_to_fit_height(5.0)
    g.next_to(sc.mobjects[1], DOWN, buff=0.5).set_x(0)
    sc.play(LaggedStart(*[FadeIn(r, shift=RIGHT * 0.15) for r in g], lag_ratio=0.2),
            run_time=2.6)

    rules = Text("RULES: no computed part-select over the configuration memory  ·  "
                 "no keep_hierarchy anywhere  ·  the XDC is implementation-only  ·  "
                 "compare a yosys estimate with the last good build before every hand-off",
                 font_size=17, color=C_RTL)
    rules.scale_to_fit_width(13.2).to_edge(DOWN, buff=0.28)
    sc.play(FadeIn(rules), run_time=0.9)
    sc.wait(2.4)


def s5_board(sc):
    sc.heading("4. Things only the board could teach us", "M11 - simulation was perfectly happy")

    a = code_block([
        "ram-readback failed on the real board.",
        "",
        "BRAM contents survive JPROGRAM. bob load wrote only up to the last",
        "non-zero word - so every zero word kept the PREVIOUS design's value.",
        "",
        "Fix: always write all 1024 words of every BRAM the design uses,",
        "and put a section in the .bit even when it is all zeros.",
    ], 20, INK)
    a[0].set_color(C_ERR)
    a[5].set_color(C_RTL); a[6].set_color(C_RTL)
    a.shift(UP * 1.4)
    sc.play(LaggedStart(*[FadeIn(l) for l in a], lag_ratio=0.18), run_time=2.0)
    sc.wait(0.8)

    b = code_block([
        "The live checks passed too - after seeing exactly ONE input vector.",
        "Eight seconds was not enough for a person to flip anything.",
        "",
        "Fix: guided checks. A description, example inputs, Enter to start,",
        "a live status line, explicit goals, and 90 seconds to reach them.",
    ], 20, INK)
    b[0].set_color(C_ERR)
    b[3].set_color(C_RTL); b[4].set_color(C_RTL)
    b.next_to(a, DOWN, buff=0.7).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in b], lag_ratio=0.18), run_time=1.9)

    r = Text("Anything a person does by hand needs a guide: what to press, what the LEDs "
             "must show, and time to do it.", font_size=19, color=C_BIT)
    r.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.3)
    sc.play(FadeIn(r), run_time=0.9)
    sc.wait(2.2)


def s6_subtle(sc):
    sc.heading("5. The quiet ones", "each of these produced a legal, plausible, wrong answer")

    rows = [
        ("route branches in sink order",
         "the .route parser attached a branch to the wrong predecessor - legal edges,"),
        ("", "wrong mux values. Only the model check caught it. Now emitted in tree order."),
        ("a Verilog macro defined twice",
         "the expected data was sent AS the stream. Unique names now."),
        ("`CNT8_BITS used above its include",
         "iverilog did not error; the register came out 2 bits wide and 298 checks failed."),
        ("a mutant pinned to a grid position",
         "carry-direct-cut cut column 5's carry; the test counter moved to column 8,"),
        ("", "then 12. The mutant survived silently - twice. It now follows the last column."),
        ("uniform random stimulus",
         "held a counter's reset half the time, so its trace was all zeros and its"),
        ("", "check could never fail. Biased 50-cycle segments now."),
    ]
    g = VGroup()
    for a, b in rows:
        if a:
            g.add(VGroup(mono(a, 18, C_ERR), Text(b, font_size=16, color=DIM))
                  .arrange(RIGHT, buff=0.4, aligned_edge=DOWN))
        else:
            g.add(Text(b, font_size=16, color=DIM))
    for r in g:
        if isinstance(r, VGroup) and len(r) == 2:
            r[1].align_to(g[0][1], LEFT).shift(RIGHT * 4.2)
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.2)
    if g.width > 13.2:
        g.scale_to_fit_width(13.2)
    g.next_to(sc.mobjects[1], DOWN, buff=0.6).set_x(0)
    sc.play(LaggedStart(*[FadeIn(r, shift=RIGHT * 0.15) for r in g], lag_ratio=0.14),
            run_time=2.8)

    note = Text("None of these threw an error. Every one was caught by comparing against "
                "something independent - a model, a trace, a mutant.",
                font_size=19, color=C_BIT)
    note.scale_to_fit_width(13.2).to_edge(DOWN, buff=0.3)
    sc.play(FadeIn(note), run_time=0.9)
    sc.wait(2.4)


def s7_rules(sc):
    sc.heading("The rules that came out of all this",
               "every one of them is in CLAUDE.md or PLAN.md, and every one was paid for")

    rules = [
        "One milestone at a time, and every milestone ends on the real board.",
        "Every Vivado build is the complete FPGA plus the new feature - never a block alone.",
        "Ground designs in tested references, and say explicitly where you diverge.",
        "Architecture numbers live in ONE file; everything else is generated from it.",
        "Committed generated data carries a stamp of what built it, and stale data is refused.",
        "Test each guard in isolation - if you can delete it and stay green, you have no test.",
        "Check that your stimulus exercises what it claims.",
        "Every hardware check runs against a stand-in board first, passing and failing.",
        "Never edit a past board result. Append.",
    ]
    g = VGroup(*[VGroup(mono("*", 19, C_BIT), Text(t, font_size=18, color=INK))
                 .arrange(RIGHT, buff=0.3, aligned_edge=DOWN) for t in rules])
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.26)
    if g.width > 13.2:
        g.scale_to_fit_width(13.2)
    g.next_to(sc.mobjects[1], DOWN, buff=0.6).set_x(0)
    sc.play(LaggedStart(*[FadeIn(r, shift=RIGHT * 0.15) for r in g], lag_ratio=0.18),
            run_time=2.8)
    sc.wait(2.6)


def s8_files(sc):
    sc.files_used(
        inputs=[("PLAN.md", "section 8: gotchas already paid for"),
                ("docs/project/REPORT.md", "section 19: all 42, with the fix"),
                ("CLAUDE.md", "the rules, in the form an agent must follow")],
        generated=[("docs/hwtest/results.log", "57 runs - the failures are still in it"),
                   ("docs/reports/M*/", "every Vivado log, including the crashes")],
        verified=[("sim/mutate_*.sh", "59 mutants - the tests that test the tests"),
                  ("tests/test_layout.py", "no Tcl in the XDC, ever again"),
                  ("tests/test_reports.py", "WNS, failing endpoints, no_clock (0)")])


EP13 = [s1_intro, s2_tcl, s3_guard, s4_vivado, s5_board, s6_subtle, s7_rules, s8_files]


class Ep13Errors(BobScene):
    def construct(self):
        self.titlecard("EPISODE 13", "Every error we hit",
                       "and the rule it became")
        for i, part in enumerate(EP13):
            part(self)
            if i < len(EP13) - 1:
                clear_all(self)


class E13S1Intro(BobScene):
    def construct(self): s1_intro(self)


class E13S2Tcl(BobScene):
    def construct(self): s2_tcl(self)


class E13S3Guard(BobScene):
    def construct(self): s3_guard(self)


class E13S4Vivado(BobScene):
    def construct(self): s4_vivado(self)


class E13S5Board(BobScene):
    def construct(self): s5_board(self)


class E13S6Subtle(BobScene):
    def construct(self): s6_subtle(self)


class E13S7Rules(BobScene):
    def construct(self): s7_rules(self)


class E13S8Files(BobScene):
    def construct(self): s8_files(self)